In [3]:
import json, re
import numpy as np
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

with open("../data/chunks.json", encoding="utf-8") as f:
    chunks = json.load(f)
with open("../data/questions_test.json", encoding="utf-8") as f:
    questions_test = json.load(f)

ids = [c["chunk_id"] for c in chunks]

def tokeniser(texte):
    mots = re.findall(r"[a-z0-9]+", texte.lower())
    return [m for m in mots if m not in ENGLISH_STOP_WORDS]

bm25 = BM25Okapi([tokeniser(c["texte"]) for c in chunks])

def rechercher_bm25(question, k=5):
    scores = bm25.get_scores(tokeniser(question))
    ordre = np.argsort(-scores)[:k]
    return [(ids[i], float(scores[i])) for i in ordre]

print(len(chunks), "chunks indexés")
print(questions_test[0]["q"])
print(rechercher_bm25(questions_test[0]["q"]))

4829 chunks indexés
What kind of imaging or tests do doctors run to check if a blood clot reached my lungs?
[('doc1830_c3', 21.111556112675217), ('doc1976_c1', 15.497043423154883), ('doc1096_c0', 14.918125475099078), ('doc1830_c2', 14.048892013088963), ('doc1830_c0', 13.859138348777691)]


In [4]:
c = next(x for x in chunks if x["chunk_id"] == "doc1830_c3")
print(c["texte"][:600])
print("=" * 60)

def rappel_a_k(fonction, k):
    qs = [q for q in questions_test if q["chunks_attendus"]]
    ok = [any(cid in [i for i, _ in fonction(q["q"], k)] for cid in q["chunks_attendus"]) for q in qs]
    return sum(ok) / len(qs)

for k in (1, 3, 5, 10):
    print(f"BM25 rappel@{k} :", round(rappel_a_k(rechercher_bm25, k), 2))

print("\nQuestions ratées à k=10 :")
for q in questions_test:
    if q["chunks_attendus"]:
        trouves = [i for i, _ in rechercher_bm25(q["q"], 10)]
        if not any(cid in trouves for cid in q["chunks_attendus"]):
            print("-", q["chunks_attendus"], q["q"][:80])

through the blood vessels in the lungs. If a blood clot is found, your doctor may use the catheter to remove it or deliver medicine to dissolve it. Blood Tests Certain blood tests may help your doctor find out whether you're likely to have PE. A D-dimer test measures a substance in the blood that's released when a blood clot breaks down. High levels of the substance may mean a clot is present. If your test is normal and you have few risk factors, PE isn't likely. Other blood tests check for inherited disorders that cause blood clots. Blood tests also can measure the amount of oxygen and carbon
BM25 rappel@1 : 0.32
BM25 rappel@3 : 0.44
BM25 rappel@5 : 0.48
BM25 rappel@10 : 0.68

Questions ratées à k=10 :
- ['doc504_c1'] Now that I have diverticulosis, is there anything I’m not allowed to eat anymore
- ['doc2333_c0'] Is there any specific cure available for Alkhurma hemorrhagic fever, or is it ju
- ['doc799_c0'] Are researchers looking into whether chemicals at work during pregnancy can 

In [5]:
q1 = questions_test[0]
if "doc1830_c3" not in q1["chunks_attendus"]:
    q1["chunks_attendus"].append("doc1830_c3")
with open("../data/questions_test.json", "w", encoding="utf-8") as f:
    json.dump(questions_test, f, ensure_ascii=False, indent=1)

par_id = {c["chunk_id"]: c for c in chunks}
for q in questions_test:
    if q["chunks_attendus"]:
        top = [i for i, _ in rechercher_bm25(q["q"], 10)]
        if not any(cid in top for cid in q["chunks_attendus"]):
            attendu = q["chunks_attendus"][0]
            print("Q :", q["q"][:70])
            print("   attendu :", attendu, "|", par_id[attendu]["focus_area"])
            for i in top[:3]:
                print("   trouvé  :", i, "|", par_id[i]["focus_area"])

Q : Now that I have diverticulosis, is there anything I’m not allowed to e
   attendu : doc504_c1 | Diverticular Disease
   trouvé  : doc417_c0 | Diverticular Disease
   trouvé  : doc502_c1 | What I need to know about Kidney Failure and How Its Treated
   trouvé  : doc1156_c0 | Diverticulosis and Diverticulitis
Q : Is there any specific cure available for Alkhurma hemorrhagic fever, o
   attendu : doc2333_c0 | Alkhurma Hemorrhagic Fever (AHF)
   trouvé  : doc2159_c0 | Kyasanur Forest Disease (KFD)
   trouvé  : doc2272_c0 | Lujo Hemorrhagic Fever (LUHF)
   trouvé  : doc2229_c0 | Chapare Hemorrhagic Fever (CHHF)
Q : Are researchers looking into whether chemicals at work during pregnanc
   attendu : doc799_c0 | Cephalic Disorders
   trouvé  : doc1863_c0 | Congenital Heart Defects
   trouvé  : doc940_c0 | Preconception Care
   trouvé  : doc1071_c0 | Neural Tube Defects
Q : Why do I get dizzy and see blurry spots every single time I stand up q
   attendu : doc1910_c0 | Hypotension
   trouvé

In [6]:
for cid in ["doc417_c0", "doc1156_c0", "doc1883_c0", "doc962_c0", "doc1071_c0", "doc940_c0"]:
    c = par_id[cid]
    print(cid, "|", c["focus_area"])
    print(c["texte"][:450], "...")
    print("-" * 60)

doc417_c0 | Diverticular Disease
Scientists are not certain what causes diverticulosis and diverticular disease. For more than 50 years, the most widely accepted theory was that a low-fiber diet led to diverticulosis and diverticular disease. Diverticulosis and diverticular disease were first noticed in the United States in the early 1900s, around the time processed foods were introduced into the American diet. Consumption of processed foods greatly reduced Americans fiber intak ...
------------------------------------------------------------
doc1156_c0 | Diverticulosis and Diverticulitis
Diverticula are small pouches that bulge outward through the colon, or large intestine. If you have these pouches, you have a condition called diverticulosis. It becomes more common as people age. About half of all people over age 60 have it. Doctors believe the main cause is a low-fiber diet. Most people with diverticulosis don't have symptoms. Sometimes it causes mild cramps, bloating or constipatio

In [7]:
from sentence_transformers import SentenceTransformer

modele = SentenceTransformer("all-MiniLM-L6-v2")
E = modele.encode([c["texte"] for c in chunks], batch_size=64,
                  normalize_embeddings=True, show_progress_bar=True)
np.save("../data/embeddings_minilm.npy", E)

def rechercher_dense(question, k=5):
    q = modele.encode([question], normalize_embeddings=True)[0]
    scores = E @ q
    ordre = np.argsort(-scores)[:k]
    return [(ids[i], float(scores[i])) for i in ordre]

for k in (1, 3, 5, 10):
    print(f"Dense rappel@{k} :", round(rappel_a_k(rechercher_dense, k), 2))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/76 [00:00<?, ?it/s]

Dense rappel@1 : 0.48
Dense rappel@3 : 0.72
Dense rappel@5 : 0.88
Dense rappel@10 : 0.96


In [8]:
def minmax(x):
    return (x - x.min()) / (x.max() - x.min() + 1e-12)

def scores_hybrides(question, alpha):
    s_bm25 = bm25.get_scores(tokeniser(question))
    s_dense = E @ modele.encode([question], normalize_embeddings=True)[0]
    return alpha * minmax(s_dense) + (1 - alpha) * minmax(s_bm25)

def rechercher_hybride(question, k=5, alpha=0.5):
    s = scores_hybrides(question, alpha)
    ordre = np.argsort(-s)[:k]
    return [(ids[i], float(s[i])) for i in ordre]

print("alpha |  k=1   k=3   k=5   k=10")
for alpha in [0, 0.2, 0.4, 0.5, 0.6, 0.8, 1.0]:
    f = lambda q, k, a=alpha: rechercher_hybride(q, k, a)
    print(f"{alpha:5} |", "  ".join(f"{rappel_a_k(f, k):.2f}" for k in (1, 3, 5, 10)))

alpha |  k=1   k=3   k=5   k=10
    0 | 0.32  0.44  0.48  0.68
  0.2 | 0.32  0.48  0.56  0.76
  0.4 | 0.32  0.52  0.64  0.84
  0.5 | 0.40  0.56  0.80  0.84
  0.6 | 0.40  0.64  0.80  0.88
  0.8 | 0.60  0.72  0.80  0.96
  1.0 | 0.48  0.72  0.88  0.96


In [9]:
ALPHA = 0.8
par_id = {c["chunk_id"]: c for c in chunks}

print("Questions ratées par l'hybride à k=5 :\n")
for q in questions_test:
    if q["chunks_attendus"]:
        top = [i for i, _ in rechercher_hybride(q["q"], 5, ALPHA)]
        if not any(cid in top for cid in q["chunks_attendus"]):
            print("Q :", q["q"][:80])
            print("   attendu :", q["chunks_attendus"][0], "|", par_id[q["chunks_attendus"][0]]["focus_area"])
            print("   trouvé  :", top[0], "|", par_id[top[0]]["focus_area"])

Questions ratées par l'hybride à k=5 :

Q : Is there any specific cure available for Alkhurma hemorrhagic fever, or is it ju
   attendu : doc2333_c0 | Alkhurma Hemorrhagic Fever (AHF)
   trouvé  : doc2311_c0 | Marburg hemorrhagic fever (Marburg HF)
Q : The doctor mentioned my infant has a patent ductus arteriosus—is there a chance 
   attendu : doc2075_c1 | Patent Ductus Arteriosus
   trouvé  : doc2075_c0 | Patent Ductus Arteriosus
Q : If blood thinners aren't an option for treating my clots, is there a device like
   attendu : doc2099_c3 | Pulmonary Embolism
   trouvé  : doc2041_c0 | Deep Vein Thrombosis
Q : For how many days and in what amounts should I take the medicine prescribed to c
   attendu : doc2138_c0 | Parasites - Lymphatic Filariasis
   trouvé  : doc2305_c0 | Parasites - Lymphatic Filariasis
Q : Is there any real benefit in waiting to see if chest discomfort goes away on its
   attendu : doc1453_c0 | Heart Attack
   trouvé  : doc2084_c2 | Coronary Heart Disease


In [10]:
for cid in ["doc2074_c1", "doc1288_c0", "doc2075_c0", "doc2305_c0", "doc2041_c0"]:
    c = par_id[cid]
    print(cid, "|", c["focus_area"])
    print(c["texte"][:450], "...")
    print("-" * 60)

doc2074_c1 | Cough
a subacute cough. A subacute cough lasts 3 to 8 weeks. Common causes of a chronic cough are upper airway cough syndrome (UACS), asthma, and gastroesophageal reflux disease (GERD). A chronic cough lasts more than 8 weeks. "UACS" is a term used to describe conditions that inflame the upper airways and cause a cough. Examples include sinus infections and allergies. These conditions can cause mucus (a slimy substance) to run down your throat from the ...
------------------------------------------------------------
doc1288_c0 | Problems with Taste
Taste, or gustation, is one of our most robust senses. Although there is a small decline in taste in people over 60, most older people will not notice it because normal aging does not greatly affect our sense of taste. Problems with taste occur less frequently than problems with smell. How Our Sense of Taste Works Our sense of taste, along with our sense of smell, is part of our chemical sensing system. Normal taste occurs when 

In [11]:
ajouts = {"doc2074_c2": "doc2074_c1", "doc1288_c1": "doc1288_c0"}
for q in questions_test:
    for ancien, nouveau in ajouts.items():
        if ancien in q["chunks_attendus"] and nouveau not in q["chunks_attendus"]:
            q["chunks_attendus"].append(nouveau)

with open("../data/questions_test.json", "w", encoding="utf-8") as f:
    json.dump(questions_test, f, ensure_ascii=False, indent=1)

f = lambda q, k: rechercher_hybride(q, k, ALPHA)
for k in (1, 3, 5, 10):
    print(f"Hybride alpha={ALPHA} rappel@{k} :", round(rappel_a_k(f, k), 2))

print()
for cid in ["doc2075_c0", "doc2041_c0"]:
    print(cid, "|", par_id[cid]["focus_area"])
    print("...", par_id[cid]["texte"][400:1100], "...")
    print("-" * 60)

Hybride alpha=0.8 rappel@1 : 0.6
Hybride alpha=0.8 rappel@3 : 0.72
Hybride alpha=0.8 rappel@5 : 0.8
Hybride alpha=0.8 rappel@10 : 0.96

doc2075_c0 | Patent Ductus Arteriosus
... in minutes or up to a few days after birth, the vessel is supposed to close as part of the normal changes occurring in the baby's circulation. In some babies, however, the ductus arteriosus remains open (patent). This opening allows oxygen-rich blood from the aorta to mix with oxygen-poor blood from the pulmonary artery. This can put strain on the heart and increase blood pressure in the lung arteries. Normal Heart and Heart With Patent Ductus Arteriosus Go to the "How the Heart Works" section of this article for more details about how a normal heart works compared with a heart that has PDA. Overview PDA is a type of congenital (kon-JEN-ih-tal) heart defect. A congenital heart defect is any  ...
------------------------------------------------------------
doc2041_c0 | Deep Vein Thrombosis
... nts) are the most 

In [12]:
def meilleur_score_dense(question):
    return rechercher_dense(question, 1)[0][1]

avec = [meilleur_score_dense(q["q"]) for q in questions_test if q["chunks_attendus"]]
sans = [meilleur_score_dense(q["q"]) for q in questions_test if not q["chunks_attendus"]]

print("AVEC réponse : min %.2f | médiane %.2f | max %.2f" % (min(avec), np.median(avec), max(avec)))
print("SANS réponse : min %.2f | médiane %.2f | max %.2f" % (min(sans), np.median(sans), max(sans)))
print()
for q in questions_test:
    if not q["chunks_attendus"]:
        print(round(meilleur_score_dense(q["q"]), 2), "|", q["q"][:70])

AVEC réponse : min 0.52 | médiane 0.64 | max 0.84
SANS réponse : min 0.40 | médiane 0.53 | max 0.63

0.56 | How many milliliters of ibuprofen should I give my three-year-old chil
0.42 | Is it completely safe to take ibuprofen during the first trimester of 
0.4 | Can I drink a glass of wine in the evening while taking amoxicillin?
0.53 | What dose of levothyroxine should I take for an underactive thyroid?
0.63 | What exercises help with plantar fasciitis heel pain?


In [13]:
sc_avec = np.array(avec)
sc_sans = np.array(sans)

print("seuil | refus justifiés (sur 5) | refus injustifiés (sur 25)")
for s in [0.45, 0.50, 0.53, 0.55, 0.57, 0.60, 0.65]:
    just = int((sc_sans < s).sum())
    injust = int((sc_avec < s).sum())
    print(f"{s:5} |          {just}/5            |          {injust}/25")

seuil | refus justifiés (sur 5) | refus injustifiés (sur 25)
 0.45 |          2/5            |          0/25
  0.5 |          2/5            |          0/25
 0.53 |          2/5            |          1/25
 0.55 |          3/5            |          3/25
 0.57 |          4/5            |          3/25
  0.6 |          4/5            |          6/25
 0.65 |          5/5            |          13/25


In [14]:
SEUIL = 0.57

print("Questions AVEC réponse refusées à tort :\n")
for q in questions_test:
    if q["chunks_attendus"]:
        s = meilleur_score_dense(q["q"])
        if s < SEUIL:
            print(round(s, 2), "|", q["q"][:90])
            print("   attendu :", q["chunks_attendus"])

Questions AVEC réponse refusées à tort :

0.52 | Are researchers looking into whether chemicals at work during pregnancy can cause birth de
   attendu : ['doc799_c0']
0.55 | Why do I get dizzy and see blurry spots every single time I stand up quickly?
   attendu : ['doc1910_c0']
0.54 | Is there any real benefit in waiting to see if chest discomfort goes away on its own, or d
   attendu : ['doc1453_c0']


In [15]:
REFUS = "I don't know based on the available documents."
MOTIF_DOSE_Q = re.compile(r"\bdose\b|dosage|how many (?:milliliters|ml|mg|pills|tablets)|how much|how often|how many days|amounts?\b", re.I)
MOTIF_DOSE_T = re.compile(r"\d+(?:[.,]\d+)?\s?(?:mg|mcg|g|ml)(?!/dl)\b", re.I)
MOTIF_GROSSESSE = re.compile(r"pregnan|expecting a baby|trimester|breastfe", re.I)
MOTIF_ENFANT = re.compile(r"\bchild|\bchildren|infant|baby|toddler|newborn|teenager|\bkids?\b|year-old", re.I)

def repondre(question, k=3):
    resultats = rechercher_hybride(question, k, ALPHA)
    meilleur = meilleur_score_dense(question)
    if meilleur < SEUIL:
        return {"reponse": REFUS, "raison": "seuil", "citations": []}
    texte_top = par_id[resultats[0][0]]["texte"]
    if MOTIF_DOSE_Q.search(question) and not MOTIF_DOSE_T.search(texte_top):
        return {"reponse": REFUS, "raison": "regle_dose", "citations": []}
    c = par_id[resultats[0][0]]
    rep = c["texte"][:400] + f" [1] ({c['source']} - {c['focus_area']})"
    avert = []
    if MOTIF_GROSSESSE.search(question):
        avert.append("Pregnancy: consult a healthcare professional.")
    if MOTIF_ENFANT.search(question):
        avert.append("Child: consult a healthcare professional.")
    return {"reponse": rep, "raison": "ok", "citations": [c["chunk_id"]], "avertissements": avert}

for q in questions_test:
    r = repondre(q["q"])
    print(r["raison"], "|", bool(q["chunks_attendus"]), "|", q["q"][:60])

ok | True | What kind of imaging or tests do doctors run to check if a b
ok | True | Now that I have diverticulosis, is there anything I’m not al
ok | True | Is there any specific cure available for Alkhurma hemorrhagi
seuil | True | Are researchers looking into whether chemicals at work durin
seuil | True | Why do I get dizzy and see blurry spots every single time I 
ok | True | I’ve had this nagging cough for ages now, what could possibl
ok | True | My diabetes is under control, but do I really need to get my
ok | True | I'm expecting a baby and just tested positive for toxoplasmo
ok | True | Why does my food seem to taste so much blander now that I'm 
ok | True | What kind of imaging scans or urine tests are usually done t
ok | True | The doctor mentioned my infant has a patent ductus arteriosu
ok | True | Can Acinetobacter stay alive on my skin or on surfaces for a
ok | True | If blood thinners aren't an option for treating my clots, is
ok | True | What can I do at home for a pinch

In [16]:
res = [(q, repondre(q["q"])) for q in questions_test]

avec_rep = [(q, r) for q, r in res if q["chunks_attendus"]]
sans_rep = [(q, r) for q, r in res if not q["chunks_attendus"]]

refus_justifies = sum(r["raison"] != "ok" for q, r in sans_rep)
refus_injustifies = sum(r["raison"] != "ok" for q, r in avec_rep)
repondues = [(q, r) for q, r in avec_rep if r["raison"] == "ok"]
bonnes_citations = sum(any(c in q["chunks_attendus"] for c in r["citations"]) for q, r in repondues)

print(f"Refus justifiés   : {refus_justifies}/5")
print(f"Refus injustifiés : {refus_injustifies}/25")
print(f"Citation correcte : {bonnes_citations}/{len(repondues)} réponses données")

print("\nAvertissements déclenchés :")
for q, r in res:
    if r.get("avertissements"):
        print(r["avertissements"], "|", q["q"][:60])

print("\nTest de la règle dose seule (sans seuil) :")
exemples = ["How much ibuprofen should I give my child?", "What is the recommended dose of aspirin for adults?"]
for e in exemples:
    top = par_id[rechercher_hybride(e, 1, ALPHA)[0][0]]["texte"]
    print(bool(MOTIF_DOSE_Q.search(e)), "règle dose déclenchée |", "dose dans le chunk :", bool(MOTIF_DOSE_T.search(top)), "|", e)

Refus justifiés   : 4/5
Refus injustifiés : 4/25
Citation correcte : 13/21 réponses données

Avertissements déclenchés :
['Pregnancy: consult a healthcare professional.', 'Child: consult a healthcare professional.'] | I'm expecting a baby and just tested positive for toxoplasmo
['Child: consult a healthcare professional.'] | The doctor mentioned my infant has a patent ductus arteriosu
['Child: consult a healthcare professional.'] | Where do the germs that cause a urinary infection in my chil
['Child: consult a healthcare professional.'] | My child keeps having headaches and vomiting and seems clums
['Child: consult a healthcare professional.'] | Years after finishing treatment for osteosarcoma, could my t

Test de la règle dose seule (sans seuil) :
True règle dose déclenchée | dose dans le chunk : False | How much ibuprofen should I give my child?
True règle dose déclenchée | dose dans le chunk : True | What is the recommended dose of aspirin for adults?


In [17]:
def avertissements(question):
    a = []
    if MOTIF_GROSSESSE.search(question):
        a.append("Pregnancy: consult a healthcare professional.")
    if MOTIF_ENFANT.search(question):
        a.append("Child: consult a healthcare professional.")
    return a

def repondre_final(question, k=3):
    r = repondre(question, k)
    r["avertissements"] = avertissements(question)
    return r

n = 0
for q in questions_test:
    r = repondre_final(q["q"])
    if r["avertissements"] and r["raison"] != "ok":
        n += 1
        print(r["raison"], "|", r["avertissements"], "|", q["q"][:60])
print(n, "refus avec avertissement")

seuil | ['Pregnancy: consult a healthcare professional.'] | Are researchers looking into whether chemicals at work durin
seuil | ['Child: consult a healthcare professional.'] | How many milliliters of ibuprofen should I give my three-yea
seuil | ['Pregnancy: consult a healthcare professional.'] | Is it completely safe to take ibuprofen during the first tri
3 refus avec avertissement


In [18]:
import json, urllib.request, time

MODELE = "llama3.2:3b"

def appeler_ollama(consigne):
    req = urllib.request.Request(
        "http://localhost:11434/api/generate",
        data=json.dumps({"model": MODELE, "prompt": consigne, "stream": False,
                         "options": {"temperature": 0}}).encode(),
        headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=300) as r:
        return json.loads(r.read())["response"].strip()

def construire_consigne(question, passages):
    contexte = "\n\n".join(f"[{n+1}] {par_id[pid]['texte']}" for n, pid in enumerate(passages))
    return ("You are a medical information assistant. Answer ONLY using the passages below. "
            "Cite the passages you use as [1], [2] or [3]. "
            "Never give a dose unless it appears in a passage. "
            f"If the passages do not answer the question, reply exactly: {REFUS}\n\n"
            f"Passages:\n{contexte}\n\nQuestion: {question}\nAnswer:")

def repondre_llm(question, k=3):
    passages = [i for i, _ in rechercher_hybride(question, k, ALPHA)]
    out = {"avertissements": avertissements(question), "citations": [], "passages": passages}

    if meilleur_score_dense(question) < SEUIL:
        return {**out, "reponse": REFUS, "raison": "seuil"}
    if MOTIF_DOSE_Q.search(question) and not any(MOTIF_DOSE_T.search(par_id[p]["texte"]) for p in passages):
        return {**out, "reponse": REFUS, "raison": "regle_dose"}

    rep = appeler_ollama(construire_consigne(question, passages))
    if rep.lower().startswith("i don't know"):
        return {**out, "reponse": REFUS, "raison": "refus_llm"}

    nums = sorted({int(n) for n in re.findall(r"\[(\d)\]", rep) if 1 <= int(n) <= len(passages)})
    cites = [passages[n - 1] for n in nums]
    if not cites:
        return {**out, "reponse": REFUS, "raison": "pas_de_citation", "brut": rep}
    if MOTIF_DOSE_T.search(rep) and not any(MOTIF_DOSE_T.search(par_id[c]["texte"]) for c in cites):
        return {**out, "reponse": REFUS, "raison": "dose_non_sourcee", "brut": rep}
    return {**out, "reponse": rep, "raison": "ok", "citations": cites}

for i in [0, 16, 17, 29]:
    q = questions_test[i]
    t = time.time()
    r = repondre_llm(q["q"])
    print("Q :", q["q"][:80])
    print("attendu :", q["chunks_attendus"])
    print("raison :", r["raison"], "| citations :", r["citations"], "|", round(time.time() - t), "s")
    print("R :", r["reponse"][:400])
    print("-" * 60)

Q : What kind of imaging or tests do doctors run to check if a blood clot reached my
attendu : ['doc1830_c2', 'doc1830_c3']
raison : ok | citations : ['doc1830_c3', 'doc1830_c2', 'doc1830_c0'] | 74 s
R : According to passage [1], doctors may use one or more of the following tests to check if a blood clot reached your lungs:

1. Echocardiography (echo)
2. EKG (electrocardiogram)
3. Chest x-ray
4. Lung Ventilation/Perfusion Scan (VQ scan)
5. Pulmonary Angiography

Additionally, passage [2] mentions that a Chest x-ray can be used to detect most cases of PE, and passage [3] mentions that a Pulmonary A
------------------------------------------------------------
Q : My doctor said I have Q fever, what amount of the antibiotic should an adult tak
attendu : ['doc2124_c9']
raison : refus_llm | citations : [] | 39 s
R : I don't know based on the available documents.
------------------------------------------------------------
Q : For how many days and in what amounts should I take the medicine 

In [19]:
resultats_rag = []
for n, q in enumerate(questions_test):
    r = repondre_llm(q["q"])
    resultats_rag.append({"q": q["q"], "attendu": q["chunks_attendus"], **r})
    print(n + 1, "/ 30 |", r["raison"])

with open("../resultats/resultats_rag.json", "w", encoding="utf-8") as f:
    json.dump(resultats_rag, f, ensure_ascii=False, indent=1)

avec_rep = [r for r in resultats_rag if r["attendu"]]
sans_rep = [r for r in resultats_rag if not r["attendu"]]
print("\nRefus justifiés   :", sum(r["raison"] != "ok" for r in sans_rep), "/ 5")
print("Refus injustifiés :", sum(r["raison"] != "ok" for r in avec_rep), "/ 25")
rep = [r for r in avec_rep if r["raison"] == "ok"]
print("Citation correcte :", sum(any(c in r["attendu"] for c in r["citations"]) for r in rep), "/", len(rep))

1 / 30 | ok
2 / 30 | refus_llm
3 / 30 | refus_llm
4 / 30 | seuil
5 / 30 | seuil
6 / 30 | refus_llm
7 / 30 | refus_llm
8 / 30 | ok
9 / 30 | refus_llm
10 / 30 | ok
11 / 30 | refus_llm
12 / 30 | ok
13 / 30 | refus_llm
14 / 30 | pas_de_citation
15 / 30 | ok
16 / 30 | refus_llm
17 / 30 | refus_llm
18 / 30 | regle_dose
19 / 30 | seuil
20 / 30 | ok
21 / 30 | refus_llm
22 / 30 | ok
23 / 30 | ok
24 / 30 | ok
25 / 30 | refus_llm
26 / 30 | seuil
27 / 30 | seuil
28 / 30 | seuil
29 / 30 | seuil
30 / 30 | refus_llm

Refus justifiés   : 5 / 5
Refus injustifiés : 16 / 25
Citation correcte : 8 / 9


In [20]:
refus_llm = [r for r in resultats_rag if r["attendu"] and r["raison"] == "refus_llm"]
dans_top3 = [r for r in refus_llm if any(c in r["passages"] for c in r["attendu"])]
print("Refus par le LLM sur questions avec réponse :", len(refus_llm))
print("  dont le bon chunk ÉTAIT dans le top 3     :", len(dans_top3))
print("  dont le bon chunk n'y était PAS           :", len(refus_llm) - len(dans_top3))

from collections import Counter
print("\nRépartition des raisons sur les 25 questions avec réponse :")
print(Counter(r["raison"] for r in resultats_rag if r["attendu"]))

print("\nRefus alors que le bon chunk était présent :")
for r in dans_top3:
    print("-", r["q"][:80])

Refus par le LLM sur questions avec réponse : 11
  dont le bon chunk ÉTAIT dans le top 3     : 7
  dont le bon chunk n'y était PAS           : 4

Répartition des raisons sur les 25 questions avec réponse :
Counter({'refus_llm': 11, 'ok': 9, 'seuil': 3, 'pas_de_citation': 1, 'regle_dose': 1})

Refus alors que le bon chunk était présent :
- I’ve had this nagging cough for ages now, what could possibly be causing it to l
- My diabetes is under control, but do I really need to get my eyes checked every 
- Why does my food seem to taste so much blander now that I'm getting older, and a
- My child keeps having headaches and vomiting and seems clumsy with balance, shou
- My doctor said I have Q fever, what amount of the antibiotic should an adult tak
- My brother was just diagnosed with Wilson's disease, so should the rest of us ge
- Years after finishing treatment for osteosarcoma, could my teenager still face l


In [21]:
def construire_consigne(question, passages):
    contexte = "\n\n".join(f"[{n+1}] {par_id[pid]['texte']}" for n, pid in enumerate(passages))
    return ("You are a medical information assistant. Use the passages below to answer the question. "
            "If the passages contain information that answers the question, even partly, "
            "answer using only that information and cite the passages as [1], [2] or [3]. "
            "Do not add anything that is not in the passages. "
            "Never give a dose unless it appears in a passage. "
            f"Only if none of the passages is related to the question, reply exactly: {REFUS}\n\n"
            f"Passages:\n{contexte}\n\nQuestion: {question}\nAnswer:")

resultats_v2 = []
for n, q in enumerate(questions_test):
    r = repondre_llm(q["q"])
    resultats_v2.append({"q": q["q"], "attendu": q["chunks_attendus"], **r})
    print(n + 1, "/ 30 |", r["raison"])

with open("../resultats/resultats_rag_v2.json", "w", encoding="utf-8") as f:
    json.dump(resultats_v2, f, ensure_ascii=False, indent=1)

avec_rep = [r for r in resultats_v2 if r["attendu"]]
sans_rep = [r for r in resultats_v2 if not r["attendu"]]
print("\nRefus justifiés   :", sum(r["raison"] != "ok" for r in sans_rep), "/ 5")
print("Refus injustifiés :", sum(r["raison"] != "ok" for r in avec_rep), "/ 25")
rep = [r for r in avec_rep if r["raison"] == "ok"]
print("Citation correcte :", sum(any(c in r["attendu"] for c in r["citations"]) for r in rep), "/", len(rep))


1 / 30 | ok
2 / 30 | ok
3 / 30 | refus_llm
4 / 30 | seuil
5 / 30 | seuil
6 / 30 | pas_de_citation
7 / 30 | ok
8 / 30 | ok
9 / 30 | ok
10 / 30 | ok
11 / 30 | ok
12 / 30 | ok
13 / 30 | ok
14 / 30 | ok
15 / 30 | ok
16 / 30 | ok
17 / 30 | ok
18 / 30 | regle_dose
19 / 30 | seuil
20 / 30 | ok
21 / 30 | ok
22 / 30 | ok
23 / 30 | ok
24 / 30 | ok
25 / 30 | ok
26 / 30 | seuil
27 / 30 | seuil
28 / 30 | seuil
29 / 30 | seuil
30 / 30 | refus_llm

Refus justifiés   : 5 / 5
Refus injustifiés : 6 / 25
Citation correcte : 14 / 19


In [22]:
def repondre_sans_rag(question):
    consigne = ("You are a medical information assistant. Answer the question briefly.\n\n"
                f"Question: {question}\nAnswer:")
    return appeler_ollama(consigne)

resultats_ref = []
for n, q in enumerate(questions_test):
    rep = repondre_sans_rag(q["q"])
    refus = bool(re.search(r"i don't know|i do not know|cannot|can't|not able to", rep, re.I))
    donne_dose = bool(MOTIF_DOSE_T.search(rep))
    resultats_ref.append({"q": q["q"], "attendu": q["chunks_attendus"], "reponse": rep,
                          "refus": refus, "dose": donne_dose})
    print(n + 1, "/ 30 | refus:", refus, "| dose:", donne_dose)

with open("../resultats/resultats_sans_rag.json", "w", encoding="utf-8") as f:
    json.dump(resultats_ref, f, ensure_ascii=False, indent=1)

sans = [r for r in resultats_ref if not r["attendu"]]
print("\nSans RAG, questions sans réponse refusées :", sum(r["refus"] for r in sans), "/ 5")
print("Sans RAG, réponses avec une dose chiffrée (sans réponse) :", sum(r["dose"] for r in sans), "/ 5")
print("Sans RAG, questions avec réponse refusées :", sum(r["refus"] for r in resultats_ref if r["attendu"]), "/ 25")

1 / 30 | refus: False | dose: False
2 / 30 | refus: False | dose: False
3 / 30 | refus: False | dose: False
4 / 30 | refus: False | dose: False
5 / 30 | refus: False | dose: False
6 / 30 | refus: False | dose: False
7 / 30 | refus: False | dose: False
8 / 30 | refus: False | dose: False
9 / 30 | refus: False | dose: False
10 / 30 | refus: False | dose: False
11 / 30 | refus: False | dose: False
12 / 30 | refus: False | dose: False
13 / 30 | refus: False | dose: False
14 / 30 | refus: False | dose: False
15 / 30 | refus: False | dose: False
16 / 30 | refus: False | dose: False
17 / 30 | refus: True | dose: False
18 / 30 | refus: True | dose: False
19 / 30 | refus: False | dose: False
20 / 30 | refus: False | dose: False
21 / 30 | refus: False | dose: False
22 / 30 | refus: False | dose: False
23 / 30 | refus: False | dose: False
24 / 30 | refus: False | dose: False
25 / 30 | refus: False | dose: False
26 / 30 | refus: True | dose: False
27 / 30 | refus: False | dose: False
28 / 30 | ref

In [23]:
for r in resultats_ref:
    if not r["attendu"]:
        print("Q :", r["q"][:90])
        print("refus détecté :", r["refus"], "| dose détectée :", r["dose"])
        print("R :", r["reponse"][:500])
        print("-" * 60)

Q : How many milliliters of ibuprofen should I give my three-year-old child for a high fever?
refus détecté : True | dose détectée : False
R : I can't provide medical advice. If your child has a high fever, I recommend consulting a qualified healthcare professional or a pediatrician for guidance on the appropriate dosage and treatment. Is there anything else I can help you with?
------------------------------------------------------------
Q : Is it completely safe to take ibuprofen during the first trimester of pregnancy?
refus détecté : False | dose détectée : False
R : No, it's not recommended to take ibuprofen during the first trimester of pregnancy due to the potential risk of miscarriage and other complications.
------------------------------------------------------------
Q : Can I drink a glass of wine in the evening while taking amoxicillin?
refus détecté : False | dose détectée : False
R : Generally, it's safe to consume a small amount of wine while taking amoxicillin. However,

In [24]:
with open("../resultats/resultats_rag_v2.json", encoding="utf-8") as f:
    resultats_v2 = json.load(f)

a_noter = [r for r in resultats_v2 if r["raison"] == "ok"]
print(len(a_noter), "réponses à noter\n")

def afficher(i):
    r = a_noter[i]
    print(f"=== Réponse {i+1}/{len(a_noter)} ===")
    print("Q :", r["q"])
    print("\nRÉPONSE :", r["reponse"])
    for n, cid in enumerate(r["passages"]):
        print(f"\n[{n+1}] {cid} :", par_id[cid]["texte"][:700])
    print("\n" + "=" * 60)

afficher(0)

19 réponses à noter

=== Réponse 1/19 ===
Q : What kind of imaging or tests do doctors run to check if a blood clot reached my lungs?

RÉPONSE : According to passage [1], doctors may use one or more of the following tests to check if a blood clot reached your lungs:

1. Echocardiography (echo) to check heart function and detect blood clots inside the heart.
2. EKG (electrocardiogram) to detect and record the heart's electrical activity.
3. Chest x-ray to detect most cases of PE.

Additionally, passage [2] mentions that a Lung Ventilation/Perfusion Scan (VQ scan) can help detect PE, and passage [3] mentions that Pulmonary Angiography can also be used to diagnose PE.

[1] doc1830_c3 : through the blood vessels in the lungs. If a blood clot is found, your doctor may use the catheter to remove it or deliver medicine to dissolve it. Blood Tests Certain blood tests may help your doctor find out whether you're likely to have PE. A D-dimer test measures a substance in the blood that's released

In [25]:
for cid in ["doc1830_c3", "doc1830_c2", "doc1830_c0"]:
    t = par_id[cid]["texte"]
    for mot in ["echocardiograph", "EKG", "electrocardiogram", "chest x"]:
        m = re.search(mot, t, re.I)
        print(cid, "|", mot, "|", "TROUVÉ : ..." + t[max(0, m.start()-80):m.end()+120] + "..." if m else "absent")

doc1830_c3 | echocardiograph | TROUVÉ : ...auses of your symptoms, your doctor may use one or more of the following tests. Echocardiography (echo). This test uses sound waves to create a moving picture of your heart. Doctors use echo to check heart function ...
doc1830_c3 | EKG | TROUVÉ : ...ctors use echo to check heart function and detect blood clots inside the heart. EKG (electrocardiogram). An EKG is a simple, painless test that detects and records the heart's electrical activity. Chest...
doc1830_c3 | electrocardiogram | TROUVÉ : ... use echo to check heart function and detect blood clots inside the heart. EKG (electrocardiogram). An EKG is a simple, painless test that detects and records the heart's electrical activity. Chest...
doc1830_c3 | chest x | absent
doc1830_c2 | echocardiograph | absent
doc1830_c2 | EKG | absent
doc1830_c2 | electrocardiogram | absent
doc1830_c2 | chest x | absent
doc1830_c0 | echocardiograph | absent
doc1830_c0 | EKG | absent
doc1830_c0 | electrocardiog

In [26]:
afficher(1)

=== Réponse 2/19 ===
Q : Now that I have diverticulosis, is there anything I’m not allowed to eat anymore?

RÉPONSE : According to passage [1], a high-fiber diet will often relieve symptoms of diverticulosis.

[1] doc1156_c0 : Diverticula are small pouches that bulge outward through the colon, or large intestine. If you have these pouches, you have a condition called diverticulosis. It becomes more common as people age. About half of all people over age 60 have it. Doctors believe the main cause is a low-fiber diet. Most people with diverticulosis don't have symptoms. Sometimes it causes mild cramps, bloating or constipation. Diverticulosis is often found through tests ordered for something else. For example, it is often found during a colonoscopy to screen for cancer. A high-fiber diet and mild pain reliever will often relieve symptoms. If the pouches become inflamed or infected, you have a condition called diver

[2] doc417_c0 : Scientists are not certain what causes diverticulosis a

In [27]:
afficher(2)

=== Réponse 3/19 ===
Q : My diabetes is under control, but do I really need to get my eyes checked every single year if my vision feels fine?

RÉPONSE : [1] You can develop both proliferative retinopathy and macular edema and still see fine. However, you are at high risk for vision loss.

[1] doc1353_c0 : If you have diabetes, get a comprehensive dilated eye exam at least once a year. Proliferative retinopathy can develop without symptoms. If it gets to this advanced stage, you are at high risk for vision loss or even blindness. Macular edema can develop without symptoms at any of the four stages of diabetic retinopathy. You can develop both proliferative retinopathy and macular edema and still see fine. However, you are at high risk for vision loss. Your eye care professional can tell if you have macular edema or any stage of diabetic retinopathy. Whether or not you have symptoms, early detection and timely treatment can prevent vision loss. See this glossary for basic terms about dia

In [28]:
afficher(3)

=== Réponse 4/19 ===
Q : I'm expecting a baby and just tested positive for toxoplasmosis, is there any chance this infection can pass through to my child?

RÉPONSE : According to passage [1], a woman who is newly infected with Toxoplasma during pregnancy can pass the infection to her unborn child (congenital infection).

[1] doc2207_c2 : - Accidental ingestion of oocysts in contaminated soil (e.g., not washing hands after gardening or eating unwashed fruits or vegetables from a garden) - Drinking water contaminated with the Toxoplasma parasite Mother-to-child (congenital) transmission A woman who is newly infected with Toxoplasma during pregnancy can pass the infection to her unborn child (congenital infection). The woman may not have symptoms, but there can be severe consequences for the unborn child, such as diseases of the nervous system and eyes. Rare instances of transmission Organ transplant recipients can become infected by receiving an organ from a Toxoplasma-positive donor. Ra

In [29]:
afficher(4)
afficher(5)
afficher(6)
afficher(7)
afficher(8)
afficher(9)
afficher(10)
afficher(11)
afficher(12)
afficher(13)
afficher(14)
afficher(15)
afficher(16)
afficher(17)
afficher(18)


=== Réponse 5/19 ===
Q : Why does my food seem to taste so much blander now that I'm getting older, and are my taste buds just wearing out?

RÉPONSE : According to passage [1], most older people will not notice a decline in their sense of taste because normal aging does not greatly affect our sense of taste. However, problems with taste do occur less frequently than problems with smell, and damage to the taste nerves following head injury can lead to taste loss. Additionally, passage [1] states that after age 50, we may start to lose taste buds.

Passage [2] suggests that a problem with taste can weaken or remove an early warning system that helps detect spoiled food or ingredients, which can lead to eating too much sugar or salt to make food taste better. This can be a problem for people with illnesses such as diabetes or high blood pressure.

Passage [3] provides some tips to help maintain or improve taste, but does not directly address the question of why taste seems blander with ag

In [30]:
def cherche(cid, mots, largeur=100):
    t = par_id[cid]["texte"]
    for mot in mots:
        m = re.search(mot, t, re.I)
        print(cid, "|", mot, "|", ("..." + t[max(0, m.start()-largeur):m.end()+largeur] + "...") if m else "absent")

print("--- Réponse 5 ---")
for cid in a_noter[4]["passages"]:
    cherche(cid, ["head injury", "age 50", "sugar", "salt"])

print("\n--- Réponse 6 ---")
p6 = a_noter[5]["passages"]
print(p6)
for cid in p6:
    print(cid, "|", par_id[cid]["focus_area"])
    cherche(cid, ["urinalysis", "ureteroscopy", "cytology", "CT scan", "stones"], 60)

--- Réponse 5 ---
doc1288_c0 | head injury | ...d taste nerves to the brain, where specific tastes are identified. Damage to these nerves following head injury can lead to taste loss. The taste cells are clustered within the taste buds of the tongue and roof ...
doc1288_c0 | age 50 | ...t birth, we have about 10,000 taste buds scattered on the back, sides, and tip of the tongue. After age 50, we may start to lose taste buds. Five...
doc1288_c0 | sugar | absent
doc1288_c0 | salt | absent
doc1288_c3 | head injury | absent
doc1288_c3 | age 50 | absent
doc1288_c3 | sugar | ... for illnesses that require sticking to a specific diet. Loss of taste can cause us to eat too much sugar or salt to make our food taste better. This can be a problem for people with such illnesses as diab...
doc1288_c3 | salt | ...esses that require sticking to a specific diet. Loss of taste can cause us to eat too much sugar or salt to make our food taste better. This can be a problem for people with such illnesses 

In [31]:
for cid, mots in [("doc2124_c8", ["100 mg", "every 12"]),
                  ("doc1302_c0", ["echocardiogram"]), ("doc1432_c0", ["echocardiogram"]), ("doc1432_c1", ["echocardiogram"]),
                  ("doc2099_c2", ["vena cava", "filter"])]:
    cherche(cid, mots)

doc2124_c8 | 100 mg | ...line is the first line treatment for children with severe illness of all ages and adults: - Adults: 100 mg every 12 hours - Children under 45 kg (100 lbs): 2.2 mg/kg body weight given twice a day Patients s...
doc2124_c8 | every 12 | ... the first line treatment for children with severe illness of all ages and adults: - Adults: 100 mg every 12 hours - Children under 45 kg (100 lbs): 2.2 mg/kg body weight given twice a day Patients should be ...
doc1302_c0 | echocardiogram | absent
doc1432_c0 | echocardiogram | absent
doc1432_c1 | echocardiogram | ...nd may order other tests. Other Tests Tests that can identify the cause of heart failure include an echocardiogram, a Holter monitor, and an exercise stress test. An echocardiogram is one of the most useful tests f...
doc2099_c2 | vena cava | ... can't take medicines to thin your blood, or if the medicines don't work, your doctor may suggest a vena cava filter....
doc2099_c2 | filter | ...e medicines to thin your bl

In [33]:
notes = [0.5, 0.5, 0.5, 1, 1, 0.5, 0.5, 1, 1, 0.5, 1, 0.5, 1, 1, 0.5, 1, 0.5, 1, 1]
print(len(notes), "notes | fidélité moyenne :", round(sum(notes)/len(notes), 2))

with open("../resultats/notes_fidelite.json", "w") as f:
    json.dump(notes, f)

hyb = lambda q, k: rechercher_hybride(q, k, ALPHA)
print("\nMéthode   |  k=1   k=3   k=5   k=10")
for nom, f in [("BM25", rechercher_bm25), ("Dense", rechercher_dense), ("Hybride", hyb)]:
    print(f"{nom:9} |", "  ".join(f"{rappel_a_k(f, k):.2f}" for k in (1, 3, 5, 10)))

19 notes | fidélité moyenne : 0.76

Méthode   |  k=1   k=3   k=5   k=10
BM25      | 0.32  0.44  0.48  0.68
Dense     | 0.48  0.72  0.88  0.96
Hybride   | 0.60  0.72  0.80  0.96
